# 1주차 실습 (해답) — 페이오프 다이어그램
『금융공학의 이해』 1장 | 대면 실습 25분

**목표**
1. 구간별 페이오프 함수를 코드로 옮기고 그림으로 확인한다.
2. 선도·옵션의 기본 포지션을 직접 그린다.
3. 교재 식 (1.8)의 분해가 성립함을 수치로 검증한다.

실행 환경: Google Colab 또는 로컬 Python (numpy, matplotlib만 사용)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True

## 예제 1 (제공) — 디지털 버퍼 노트의 페이오프
V1에서 본 J.P. Morgan 디지털 버퍼 노트 (교재 식 1.1).
최초 기준지수 $S_0 = 100$, 기준선 $K = 0.85 S_0 = 85$.

In [ ]:
S0 = 100.0
K_note = 0.85 * S0   # 이 상품의 기준선 (연습의 행사가격 K와 혼동 금지)

def note_payoff(ST):
    """디지털 버퍼 노트의 만기 지급액 (교재 식 1.1)."""
    ST = np.asarray(ST, dtype=float)
    down = 1000 + 1000 * ((ST - S0) / S0 + 0.15) / 0.85
    return np.where(ST >= K_note, 1076.90, down)

ST = np.linspace(0, 150, 1501)
plt.plot(ST, note_payoff(ST), lw=2)
plt.axvline(K_note, ls=":", c="gray")
plt.xlabel("S_T"); plt.ylabel("payoff"); plt.title("Digital Buffered Note")
plt.show()

# 교재 표 1.1의 값 재현
for st in [85.0, 84.99, 70.0, 150.0]:
    print(f"S_T = {st:>7.2f} -> {note_payoff(st):>9.2f}")

## 연습 1 — 선도계약의 long과 short
인도가격 $K=100$인 선도계약의 두 포지션 $S_T - K$, $K - S_T$를
한 좌표에 겹쳐 그려 네 구간을 확인하라 (교재 그림 1.3).

In [ ]:
K = 100.0
ST = np.linspace(0, 200, 401)

long_fwd  = ST - K
short_fwd = K - ST

plt.plot(ST, long_fwd, lw=2, label="long forward: $S_T-K$")
plt.plot(ST, short_fwd, lw=2, ls="--", label="short forward: $K-S_T$")
plt.axvline(K, ls=":", c="gray"); plt.axhline(0, c="k", lw=0.5)
plt.xlabel("S_T"); plt.ylabel("payoff"); plt.legend()
plt.show()

## 연습 2 — 옵션의 네 가지 포지션
행사가격 $K=100$으로 long call, short call, long put, short put의
만기 페이오프를 $2\times2$ 격자에 그려라 (교재 그림 1.4, 표 1.3).

| 포지션 | 페이오프 |
|---|---|
| long call | $\max(S_T-K,0)$ |
| short call | $-\max(S_T-K,0)$ |
| long put | $\max(K-S_T,0)$ |
| short put | $-\max(K-S_T,0)$ |

In [ ]:
def long_call(ST, K):  return np.maximum(ST - K, 0)
def short_call(ST, K): return -np.maximum(ST - K, 0)
def long_put(ST, K):   return np.maximum(K - ST, 0)
def short_put(ST, K):  return -np.maximum(K - ST, 0)

positions = [("long call", long_call), ("long put", long_put),
             ("short call", short_call), ("short put", short_put)]

fig, axes = plt.subplots(2, 2, figsize=(9, 6), sharex=True)
for ax, (name, f) in zip(axes.ravel(), positions):
    ax.plot(ST, f(ST, K), lw=2)
    ax.axvline(K, ls=":", c="gray"); ax.axhline(0, c="k", lw=0.5)
    ax.set_title(name)
fig.tight_layout(); plt.show()

## 연습 3 — 분해식 검증 (교재 식 1.8)
디지털 버퍼 노트는 다음 세 조각으로 분해된다 ($K = 0.85 S_0$):

$$X_T = \frac{1000}{K} S_T - \frac{1000}{K}\max(S_T-K,0) + 76.90\,\mathbb{1}_{\{S_T \ge K\}}$$

지수 비례 포지션, short call, 디지털 옵션의 합이다.
이 분해가 예제 1의 `note_payoff`와 **모든** $S_T$에서 일치하는지 수치로 검증하라.

In [ ]:
def note_decomposed(ST):
    ST = np.asarray(ST, dtype=float)
    piece1 = (1000 / K_note) * ST
    piece2 = -(1000 / K_note) * np.maximum(ST - K_note, 0)
    piece3 = 76.90 * (ST >= K_note)
    return piece1 + piece2 + piece3

ST = np.linspace(0, 150, 1501)
print("모든 점에서 일치:", np.allclose(note_decomposed(ST), note_payoff(ST)))

## 연습 4 — 페이오프와 이익의 구분 (연습문제 1.4)
현재 가격 $C_0 = 5$, 행사가격 $K=100$, 만기 1년 콜옵션. 만기 주가 $S_T = 120$,
현재 1원의 만기가치 $R_{0,T} = 1.05$.

(a) long call의 만기 페이오프, (b) 만기 기준 이익을 계산하고,
(c) 왜 단순히 $20 - 5$라고 하면 안 되는지 한 줄로 설명하라.

In [ ]:
C0, K4, ST_val, R = 5.0, 100.0, 120.0, 1.05

payoff = max(ST_val - K4, 0)          # (a) = 20
profit_T = payoff - C0 * R           # (b) = 20 - 5.25 = 14.75
print(f"(a) 만기 페이오프 = {payoff}")
print(f"(b) 만기 기준 이익 = {profit_T}")
print("(c) C0=5는 현재 시점의 돈이므로 만기 기준으로는 그 만기가치 5.25를 빼야 한다.")

## 조별 활동 (10분) — 실제 상품 분해하기
#[LOCAL] 한국 시장 예시 — 번역판에서는 현지 상품으로 교체

증권사 ELS(주가연계증권) 안내문을 하나 골라, 오늘 배운 방법으로 분해해 보자.
1. 미래 상태를 몇 개로 나누면 되는가?
2. 각 상태의 지급액은 얼마인가? (구간별 함수로 적기)
3. 어떤 기본 조각(지수 비례, 콜/풋, 디지털)의 합으로 보이는가?

다음 주 퀴즈에 이 활동에서 고른 상품이 한 문제 나온다.